In [86]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Actividades

Mendoza Hernández Carlos Emiliano

1. Basándose en la información contenida en el archivo `plane_routes.csv`, realice una infografía de las operaciones realizadas por los aeropuertos del mundo así como de las rutas aéreas seguidas por las aeronaves. La infografía deberá mostrar:

    a. Las operaciones realizadas por cada aeropuerto, georeferenciado.

    b. El trazo de las rutas seguidas por las aeronaves

    c. Las 20 aeronaves más usadas a nivel global.

### Preproceso de los datos

In [87]:
df1 = pd.read_csv('./data/plane_routes.csv', usecols=['source airport', 'destination airport', 'equipment'])
df1.dropna(subset=['source airport', 'destination airport'], inplace=True)
df1

,source airport,destination airport,equipment
0,AER,KZN,CR2
1,ASF,KZN,CR2
2,ASF,MRV,CR2
3,CEK,KZN,CR2
4,CEK,OVB,CR2
...,...,...,...
67658,WYA,ADL,SF3
67659,DME,FRU,734
67660,FRU,DME,734
67661,FRU,OSS,734


In [88]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67663 entries, 0 to 67662
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   source airport       67663 non-null  object
 1   destination airport  67663 non-null  object
 2   equipment            67645 non-null  object
dtypes: object(3)
memory usage: 1.5+ MB


In [89]:
df2 = pd.read_excel('./data/GlobalAirportDatabase.xlsx', usecols=['IATA Code', 'Name', 'City/Town', 'Country', 'Latitude Decimal Degrees', 'Longitude Decimal Degrees'])
df2.dropna(subset=['IATA Code'], inplace=True)
df2.dropna(subset=['Latitude Decimal Degrees', 'Longitude Decimal Degrees'], inplace=True)
df2 = df2[(df2['Latitude Decimal Degrees'] != 0.0) & (df2['Longitude Decimal Degrees'] != 0.0)]
df2

,IATA Code,Name,City/Town,Country,Latitude Decimal Degrees,Longitude Decimal Degrees
0,GKA,GOROKA,GOROKA,PAPUA NEW GUINEA,-6.082,145.392
2,MAG,MADANG,MADANG,PAPUA NEW GUINEA,-5.207,145.789
3,HGU,MOUNT HAGEN,MOUNT HAGEN,PAPUA NEW GUINEA,-5.826,144.296
4,LAE,NADZAB,NADZAB,PAPUA NEW GUINEA,-6.570,146.726
5,POM,PORT MORESBY JACKSONS INTERNATIONAL,PORT MORESBY,PAPUA NEW GUINEA,-9.443,147.220
...,...,...,...,...,...,...
9275,KHG,KASHI,KASHI,CHINA,39.543,76.022
9276,HTN,HOTAN,HOTAN,CHINA,37.038,79.866
9278,URC,DIWOPU,URUMQI,CHINA,43.907,87.474
9286,HRB,TAIPING,HARBIN,CHINA,45.623,126.250


In [90]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2948 entries, 0 to 9296
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   IATA Code                  2948 non-null   object 
 1   Name                       2948 non-null   object 
 2   City/Town                  2948 non-null   object 
 3   Country                    2948 non-null   object 
 4   Latitude Decimal Degrees   2948 non-null   float64
 5   Longitude Decimal Degrees  2948 non-null   float64
dtypes: float64(2), object(4)
memory usage: 161.2+ KB


In [91]:
# Merge for source airport details with coordinates and location info
merged = df1.merge(
    df2[['IATA Code', 'Name', 'City/Town', 'Country', 
          'Latitude Decimal Degrees', 'Longitude Decimal Degrees']],
    left_on='source airport',
    right_on='IATA Code',
    how='inner'
).rename(columns={
    'Name': 'source airport name',
    'City/Town': 'source city',
    'Country': 'source country',
    'Latitude Decimal Degrees': 'source lat',
    'Longitude Decimal Degrees': 'source lon'
}).drop(columns='IATA Code')

# Merge for destination airport details with coordinates and location info
merged = merged.merge(
    df2[['IATA Code', 'Name', 'City/Town', 'Country',
          'Latitude Decimal Degrees', 'Longitude Decimal Degrees']],
    left_on='destination airport',
    right_on='IATA Code',
    how='inner'
).rename(columns={
    'Name': 'destination airport name',
    'City/Town': 'destination city',
    'Country': 'destination country',
    'Latitude Decimal Degrees': 'destination lat',
    'Longitude Decimal Degrees': 'destination lon'
}).drop(columns='IATA Code')

# Select and order final columns
final_df = merged[[
    'source airport', 'source airport name',
    'source city', 'source country',
    'source lat', 'source lon',
    'destination airport', 'destination airport name',
    'destination city', 'destination country',
    'destination lat', 'destination lon',
    'equipment'
]]
final_df.to_csv('./data/plane_routes_with_coordinates.csv', index=False)
final_df

,source airport,source airport name,source city,source country,source lat,source lon,destination airport,destination airport name,destination city,destination country,destination lat,destination lon,equipment
0,AER,SOCHI,SOCHI,RUSSIA,43.446,39.947,KZN,KAZAN,KAZAN,RUSSIA,55.608,49.277,CR2
1,ASF,ASTRAKHAN,ASTRAKHAN,RUSSIA,46.283,48.006,KZN,KAZAN,KAZAN,RUSSIA,55.608,49.277,CR2
2,ASF,ASTRAKHAN,ASTRAKHAN,RUSSIA,46.283,48.006,MRV,MINERALNYYE VODY,MINERALNYE VODY,RUSSIA,44.225,43.082,CR2
3,CEK,BALANDINO,CHELYABINSK,RUSSIA,55.303,61.507,KZN,KAZAN,KAZAN,RUSSIA,55.608,49.277,CR2
4,KZN,KAZAN,KAZAN,RUSSIA,55.608,49.277,AER,SOCHI,SOCHI,RUSSIA,43.446,39.947,CR2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50698,TSV,TOWNSVILLE,TOWNSVILLE,AUSTRALIA,-19.253,146.765,ISA,MOUNT ISA,MOUNT ISA,AUSTRALIA,-20.664,139.489,SF3
50699,WGA,WAGGA WAGGA,WAGGA WAGGA,AUSTRALIA,-35.165,147.466,MEL,MELBOURNE INTERNATIONAL,MELBOURNE,AUSTRALIA,-37.673,144.843,SF3
50700,WGA,WAGGA WAGGA,WAGGA WAGGA,AUSTRALIA,-35.165,147.466,SYD,KINGSFORD SMITH INTERNATIONAL AIRPORT,SYDNEY,AUSTRALIA,-33.946,151.177,SF3
50701,FRU,MANAS,BISHKEK,RUSSIA,43.062,74.478,OSS,OSH,OSH,RUSSIA,40.609,72.793,734


### Grafica

In [93]:
# Filter coordinates with new column names
map_df = final_df.dropna(subset=[
    'source lat', 'source lon', 
    'destination lat', 'destination lon'
]).sample(200)

# Create base map
fig = go.Figure()

# Add airport markers (both source and destination)
fig.add_trace(go.Scattergeo(
    lon = map_df['source lon'].tolist() + map_df['destination lon'].tolist(),
    lat = map_df['source lat'].tolist() + map_df['destination lat'].tolist(),
    mode = 'markers',
    marker = dict(
        size = 4,
        color = '#1f77b4'
        # opacity = 0.7
    ),
    name = 'Airports',
    hovertext = map_df['source airport name'].tolist() + map_df['destination airport name'].tolist()
))

# Add flight paths with new column names
for _, row in map_df.iterrows():
    fig.add_trace(go.Scattergeo(
        lon = [row['source lon'], row['destination lon']],
        lat = [row['source lat'], row['destination lat']],
        mode = 'lines',
        line = dict(
            width = 0.7,
            color = '#ff7f0e'
            # opacity = 0.3
        ),
        hoverinfo = 'text',
        text = f"From: {row['source airport name']}<br>To: {row['destination airport name']}",
        showlegend = False
    ))

# Update layout
fig.update_layout(
    title_text = 'Global Flight Routes Visualization',
    geo = dict(
        showland = True,
        landcolor = 'lightgray',
        countrycolor = 'white',
        projection_type = 'natural earth',
        coastlinewidth = 0.5
    ),
    margin = dict(l=0, r=0, t=40, b=0)
)

fig.show()